In [2]:
import pandas as pd
import numpy as np
from gspread_pandas import Spread, conf

In [3]:
# --- 1. CONFIGURATION & CONNECTION ---
config_path = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
emarath_global_sheet_url = "https://docs.google.com/spreadsheets/d/1aXobnCl4QQ-hQ_IWoTPo2ZEo3HM5kNwycwMBJtEDOfc/edit?gid=822921427#gid=822921427"
daily_report_emarath_global = "https://docs.google.com/spreadsheets/d/1Vwq5UWy66o47ND5N3RKWsjIGtz3ybvsfZU5HnmLXsuU/edit?gid=2059217672#gid=2059217672"
c = conf.get_config(file_name=config_path)


In [4]:
logistic_sheet_url = "https://docs.google.com/spreadsheets/d/1eo2tY_57lcOTtGOyU5GOz2IQON3b0VsVf8rTsbfT3HM/edit?gid=0#gid=0"
logi_spread = Spread(logistic_sheet_url, config=c)

In [3]:
def safe_upload(spreadsheet, df, sheet_name):
   
    if not df.empty:
        spreadsheet.open_sheet(sheet_name, create=True)

        required_rows = len(df) + 2
        required_cols = len(df.columns)

        current_rows = spreadsheet.sheet.row_count
        current_cols = spreadsheet.sheet.col_count

        if current_rows < required_rows or current_cols < required_cols:
            spreadsheet.sheet.resize(rows=max(required_rows, 100), cols=max(required_cols, 26))

        last_col_letter = chr(64 + min(current_cols, 26))
        spreadsheet.sheet.batch_clear([f'A2:{last_col_letter}{max(current_rows, 2)}'])

        spreadsheet.df_to_sheet(df, index=False, replace=False, headers=False, start="A2")
        print(f"Successfully updated sheet: {sheet_name}")

In [7]:
logi_df = logi_spread.sheet_to_df(sheet="Order list", index=None)
logi_df.head(2)

,SL NO,AGENT,DATE,TRACKING NUMBER,EM NUMBER,NAME,NUMBER1,NUMBER2,STATE/CITY,ADDRESS,...,COUNTRY,DELIVERY AGENT,DISPATCHED DATE,STATUS,DELIVERED/CANCELLED DATE,DELAYED REASON,CANCELLATION REASON,ZONE,WILAYAT,NOTES
0,0,EMARATH GLOBAL,31/01/2026,2602060108039,EM85914,Charles Rocky,502781241,502781241,Dubail,Yateem Group building.19 b street. Al Quoz 4,...,UAE,TAWSEEL,31/01/2026,Delivered and unpaid,03/02/2026,,,,,
1,1,AMINA,31/01/2026,397713669,EMC264,JABIR,547646770,9660506645614,Jeddah,Saudi Arabia\n Jeddah\n Al safa 9\n Umm Al Qura,...,KSA,NAQEL,04/02/2026,Delivered and unpaid,,,,,S31821,


In [31]:
logi_df.columns

Index(['SL NO', 'AGENT', 'DATE', 'TRACKING NUMBER', 'EM NUMBER', 'NAME',
       'NUMBER1', 'NUMBER2', 'STATE/CITY', 'ADDRESS', 'PRODUCT1', 'QTY',
       'PRODUCT2', 'QTY', 'TOTAL', 'PAYMENT METHOD', 'REMARKS',
       'NATIONAL CODE', 'COUNTRY', 'DELIVERY AGENT', 'DISPATCHED DATE',
       'STATUS', 'DELIVERED/CANCELLED DATE', 'DELAYED REASON',
       'CANCELLATION REASON', 'ZONE', 'WILAYAT', 'NOTES', 'DATE_ONLY'],
      dtype='str')

In [20]:
logi_df = logi_df.dropna(subset=['TRACKING NUMBER']).copy()
logi_df = logi_df[logi_df['TRACKING NUMBER'].astype(str).str.strip() != ""]
logi_df['DATE'] = pd.to_datetime(logi_df['DATE'], errors='coerce', dayfirst=True)
logi_df = logi_df.dropna(subset=['DATE']) 
            
          
logi_df['DATE_ONLY'] = logi_df['DATE'].dt.date
logi_report = logi_df.groupby(['DATE_ONLY','AGENT']).agg(
    ORDER_CONVERTED=('STATUS', 'count')
).reset_index()


In [21]:
logi_report

,DATE_ONLY,AGENT,ORDER_CONVERTED
0,2026-01-31,AMINA,2
1,2026-01-31,ARUN,1
2,2026-01-31,BURHANA,5
3,2026-01-31,CHAITHANYA,2
4,2026-01-31,EMARATH GLOBAL,1
...,...,...,...
205,2026-02-23,SUHANTHIKA,1
206,2026-02-23,Safwan,2
207,2026-02-23,ZAKIYA,4
208,2026-02-24,RAHIB,1


In [22]:

target_bde_sheets = [
    'RAHIB', 'HABIYA', 'BURHANA', 'SHAMNA', 'ARUN', 
    'CHAITHANYA', 'ZAKIYA', 'AJIN', 'SAFAN', 'SUSHANTHIKA', 
    'ADWAITHA', 'NEHA', 'GOWTHAM', 'AMINA', 'ADHIL', 
    'SHABNA', 'FARSANA', 'RINSIYA', 'ARSHAD', 'NAJA', 'NAFI'
]

# lead_statuses = ["WON", "SUPER HOT", "HOT", "WARM", "COLD", "BOOKING", "WHATS APP ENGAGE"]


emarath_spread = Spread(emarath_global_sheet_url, config=c)


all_data_list = []

def safe_upload(spreadsheet, df, sheet_name):
    if not df.empty:
        spreadsheet.open_sheet(sheet_name, create=True)
        spreadsheet.sheet.batch_clear(['A2:Z2000']) 
        spreadsheet.df_to_sheet(df, index=False, replace=False, headers=False, start="A2")


for sheet_name in target_bde_sheets:
    try:
        df = emarath_spread.sheet_to_df(sheet=sheet_name, index=None, header_rows=1)
        
        mapping_cols = {
            'AGENT'       : 'AGENT',
            'COUNTRY'     : 'REGION',
            'CUSTOMER PATH': 'CUSTOMER PATH',
            'PRODUCT 1'   : 'PRODUCT',
            'NAME'        : 'NAME',
            'PHONE NO 1'  : 'PHONE NO',
            'STATUS'      : 'STATUS',
            'DATE'        : 'DATE',
            'VALUE'       : 'AMOUNT'
        }
        
        if not df.empty and all(col in df.columns for col in mapping_cols.keys()):
            subset = df[list(mapping_cols.keys())].copy()
            subset.rename(columns=mapping_cols, inplace=True)

            subset['DATE'] = pd.to_datetime(subset['DATE'], errors='coerce', dayfirst=True)
            subset = subset.dropna(subset=['DATE']) 
            
            subset['DATE_ONLY'] = subset['DATE'].dt.date
            
            invalid_agents = ['', 'NAN', 'NONE', 'N/A', 'NA']
            subset = subset[~subset['AGENT'].isin(invalid_agents)]

            subset['AMOUNT'] = pd.to_numeric(subset['AMOUNT'], errors='coerce').fillna(0)
            for col in ['CUSTOMER PATH', 'STATUS', 'AGENT']:
                subset[col] = subset[col].astype(str).str.strip().str.upper()

            all_data_list.append(subset)
                
    except Exception as e:
        print(f"Error in sheet {sheet_name}: {e}")



if all_data_list:
    master_df = pd.concat(all_data_list, ignore_index=True)
    master_df['DATE_ONLY'] = master_df['DATE'].dt.date

    invalid_phones = ['0', 'NAN', 'NONE', '', 'nan', 'None']
    master_df['is_contacted'] = (
        (master_df['CUSTOMER PATH'] == 'LEAD') & 
        (~master_df['PHONE NO'].astype(str).str.strip().isin(invalid_phones)) &
        (master_df['PHONE NO'].notna())
    )
    master_df['is_pending'] = (
        (master_df['CUSTOMER PATH'] == 'LEAD') & 
        ((master_df['PHONE NO'].astype(str).str.strip().isin(invalid_phones)) | (master_df['PHONE NO'].isna()))
    )
    master_df['is_order_conv'] = (master_df['CUSTOMER PATH'] == 'LEAD') & (master_df['STATUS'] == 'WON')
    master_df['is_followup_conv'] = (master_df['CUSTOMER PATH'] == 'MISSED LEAD') & (master_df['STATUS'] == 'WON')

    # Aggregate using the new helper columns ---
    report_data = master_df.groupby(['DATE_ONLY', 'AGENT']).agg(
        TOTAL_LEAD=('CUSTOMER PATH', lambda x: (x == 'LEAD').sum()),
        CONTACTED=('is_contacted', 'sum'),
        # ORDER_CONVERTED=('is_order_conv', 'sum'),
        PENDING=('is_pending', 'sum'),
        FOLLOW_UP_LEADS=('CUSTOMER PATH', lambda x: (x == 'MISSED LEAD').sum()),
        FOLLOW_UP_LEADS_CONVERTED=('is_followup_conv', 'sum'),
        SALES_AMOUNT=('AMOUNT', 'sum')
    ).reset_index()

   


    file_path = "./Output_Data/BDE_Report.xlsx"
    
    

else:
    print("No data found to process.")

In [5]:
report_data.to_excel(file_path)

In [24]:
report_data.head(2)

,DATE_ONLY,AGENT,TOTAL_LEAD,CONTACTED,PENDING,FOLLOW_UP_LEADS,FOLLOW_UP_LEADS_CONVERTED,SALES_AMOUNT
0,2026-02-04,ADHIL,13,13,0,3,0,180.0
1,2026-02-04,AMINA,36,36,0,0,0,350.0


In [27]:
report_data['AGENT'] = report_data['AGENT'].str.strip().str.upper()
logi_report['AGENT'] = logi_report['AGENT'].str.strip().str.upper()

merged_df = report_data.merge(
    logi_report[['DATE_ONLY', 'AGENT', 'ORDER_CONVERTED']],
    on=['DATE_ONLY', 'AGENT'],
    how='left'
)

merged_df['ORDER_CONVERTED'] = merged_df['ORDER_CONVERTED'].fillna(0).astype(int)
merged_cols_order = ['DATE_ONLY', 'AGENT', 'TOTAL_LEAD', 'CONTACTED', 'ORDER_CONVERTED', 'PENDING', 'FOLLOW_UP_LEADS','FOLLOW_UP_LEADS_CONVERTED','SALES_AMOUNT']
merged_df = merged_df[[c for c in merged_cols_order if c in merged_df.columns]]

merged_df.head()

,DATE_ONLY,AGENT,TOTAL_LEAD,CONTACTED,ORDER_CONVERTED,PENDING,FOLLOW_UP_LEADS,FOLLOW_UP_LEADS_CONVERTED,SALES_AMOUNT
0,2026-02-04,ADHIL,13,13,1,0,3,0,180.0
1,2026-02-04,AMINA,36,36,0,0,0,0,350.0
2,2026-02-04,ARUN,34,34,0,0,1,1,360.0
3,2026-02-04,BURHANA,30,30,1,0,10,0,170.0
4,2026-02-04,CHAITHANYA,41,41,2,0,1,0,710.0


In [28]:
import math

report_data_df = merged_df.copy()
report_data_df['WEEK'] = pd.to_datetime(report_data_df['DATE_ONLY']).apply(
    lambda x: f"WEEK {math.ceil(x.day / 7)}"
)


cols_order = ['DATE_ONLY', 'WEEK', 'AGENT', 'TOTAL_LEAD', 'CONTACTED', 'ORDER_CONVERTED', 'PENDING', 'FOLLOW_UP_LEADS','FOLLOW_UP_LEADS_CONVERTED','SALES_AMOUNT']
report_data_df = report_data_df[[c for c in cols_order if c in report_data_df.columns]]


weekly_summary = report_data_df.groupby(['WEEK', 'AGENT']).agg({
    'DATE_ONLY': lambda x: f"{x.min().strftime('%d/%m')} - {x.max().strftime('%d/%m')}",
    'TOTAL_LEAD': 'sum',
    'CONTACTED': 'sum',
    'ORDER_CONVERTED': 'sum',
    'PENDING': 'sum',
    'FOLLOW_UP_LEADS' : 'sum',
    'FOLLOW_UP_LEADS_CONVERTED' : 'sum',
    'SALES_AMOUNT': 'sum'
}).reset_index()

weekly_summary.rename(columns={'DATE_ONLY': 'DATE_RANGE'}, inplace=True)
cols_order = ['DATE_RANGE', 'WEEK', 'AGENT', 'TOTAL_LEAD', 'CONTACTED', 'ORDER_CONVERTED', 
              'PENDING', 'FOLLOW_UP_LEADS', 'FOLLOW_UP_LEADS_CONVERTED', 'SALES_AMOUNT']
weekly_summary = weekly_summary[[c for c in cols_order if c in weekly_summary.columns]]

daily_report_spread = Spread(daily_report_emarath_global, config=c)
# Upload Daily Data to 'DATA' sheet
# safe_upload(daily_report_spread, merged_df, 'Daily_Report')
    
# Upload Weekly Summary to 'Weekly_Summary_Report'
# safe_upload(daily_report_spread, weekly_summary, 'Weekly_Summary_Report')

print("Reports updated successfully with Weekly categorization!")

Reports updated successfully with Weekly categorization!


In [30]:
weekly_summary

,DATE_RANGE,WEEK,AGENT,TOTAL_LEAD,CONTACTED,ORDER_CONVERTED,PENDING,FOLLOW_UP_LEADS,FOLLOW_UP_LEADS_CONVERTED,SALES_AMOUNT
0,04/02 - 07/02,WEEK 1,ADHIL,83,83,7,0,7,2,1610.0
1,06/02 - 07/02,WEEK 1,AJIN,28,28,1,0,0,0,180.0
2,04/02 - 07/02,WEEK 1,AMINA,121,121,7,0,4,3,1600.0
3,04/02 - 07/02,WEEK 1,ARUN,113,113,5,0,4,4,1430.0
4,04/02 - 07/02,WEEK 1,BURHANA,113,113,7,0,18,1,1430.0
...,...,...,...,...,...,...,...,...,...,...
65,23/02 - 25/02,WEEK 4,RINSIYA,57,57,0,0,5,2,360.0
66,23/02 - 25/02,WEEK 4,SAFAN,60,59,0,1,11,5,1660.0
67,24/02 - 25/02,WEEK 4,SHAMNA,35,35,0,0,0,0,0.0
68,23/02 - 25/02,WEEK 4,SUHANTHIKA,69,60,1,9,2,1,120.0
